# IEC 60870-5-104 — substations and RTUs

A SCADA master and a simulated 20 kV feeder bay RTU talk over an in-memory TCP link. Replace `UseInMemory(rtuListener)` with `UseTcp("10.0.0.5")` to reach a real RTU — commands only on equipment you are authorised to operate.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.19.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.19.0-preview.1"

## APDUs on the wire
U frames start and test the link, S frames acknowledge, I frames carry numbered ASDUs.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Iec104;
using IoTCom.Net.Transports;

Console.WriteLine(Iec104Apdu.Parse(Convert.FromHexString("680407000000")));
Console.WriteLine(Iec104Apdu.Parse(Convert.FromHexString("680401000400")));
var giFrame = Convert.FromHexString("680E0000000064010600010000000014");
Console.WriteLine(Iec104Apdu.Parse(giFrame));
foreach (var f in Iec104Apdu.Describe(giFrame)) Console.WriteLine($"{f.Name,-8} {f.Value}");

## Connect and interrogate
STARTDT, then a general interrogation returns every value; counters come with a counter interrogation.

In [ ]:
var rtuListener = new InMemoryTransportListener("notebook-rtu");
var rtu = Iec104SubstationSimulator.Create(o => { o.ListenInMemory(rtuListener); o.RequireSelectBeforeOperate = true; });
await rtu.StartAsync();
var scada = Iec104Client.Create(o => o.UseInMemory(rtuListener).AllowCommands());
await scada.ConnectAsync();
foreach (var p in (await scada.InterrogateAsync()).OrderBy(p => p.Object.Address)) Console.WriteLine(p);
Console.WriteLine((await scada.CounterInterrogateAsync()).Single());

## Spontaneous, time-tagged changes
After the interrogation the RTU only sends changes, each with a CP56Time2a time tag.

In [ ]:
var changes = new System.Collections.Concurrent.ConcurrentQueue<Iec104PointValue>();
scada.PointReceived += p => { if (p.Cause == Iec104Cause.Spontaneous) changes.Enqueue(p); };
rtu.Trip();
await Task.Delay(300);
foreach (var c in changes.Where(c => c.Object.Address < 2000)) Console.WriteLine(c);

## Select-before-operate and interlocks
The breaker needs select then execute; the RTU refuses to close while the protection trip is latched.

In [ ]:
try { await scada.DoubleCommandAsync(Iec104SubstationSimulator.Ioa.BreakerCommand, on: true, selectBeforeOperate: true); }
catch (IoTCom.Net.DeviceException e) { Console.WriteLine("refused: " + e.Message); }
await scada.SingleCommandAsync(Iec104SubstationSimulator.Ioa.TripReset, true, selectBeforeOperate: true);
var term = await scada.DoubleCommandAsync(Iec104SubstationSimulator.Ioa.BreakerCommand, on: true, selectBeforeOperate: true);
Console.WriteLine(term);
await Task.Delay(200);
Console.WriteLine(scada.Points[(1, Iec104SubstationSimulator.Ioa.Breaker)]);
await scada.DisposeAsync();
await rtu.DisposeAsync();

## Going further
`iotcom iec104 gi --sim`, `iotcom iec104 monitor --sim`, `iotcom iec104 command double 5001 off --sbo --sim --allow-write`, `iotcom iec104 serve`, the Gallery's *Substation control over IEC 104* and the Iec104Scada sample. See `docs/en/protocols/iec104.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*